# 05. Context, memory, and the agent harness

Priya wants a March briefing that compares February, names the leading region, and notes what the report includes. She also asks for all future briefings in two bullets. Later she returns to the same chat and says, “Write the briefing now.”

Start Ollama and the [sales API](04_sales_agent_api.md) before running the cells. The notebook uses `SALES_API_URL`, which defaults to `http://127.0.0.1:8000`. The API can use either provider. The briefing cell calls Ollama directly.

As you go, decide which facts belong to the conversation, which belong to Priya's profile, and which should come from the sales report.

## 1. Send the latest message by itself

The helper sends a JSON question to the API and returns the answer. `raise_for_status()` raises an error for an unsuccessful HTTP response. Compare the replies to the complete March question and “Write the briefing now.”

In [ ]:
import os

import httpx

API_URL = os.environ.get("SALES_API_URL", "http://127.0.0.1:8000")


def ask_api(route, question):
    response = httpx.post(
        API_URL + route,
        json={"question": question},
        timeout=40,
    )
    response.raise_for_status()
    return response.json()["answer"]


print(ask_api("/chat", "Which region had the highest sales in March?"))
print(ask_api("/chat", "Write the briefing now."))

The second request asks which month to check. The API only received the latest text. To continue Priya's task, the caller needs to supply the earlier decision.

## 2. Store conversation state and preferences

| Information | Example | Where to look it up |
|---|---|---|
| Conversation state | Compare March with February. | By user and conversation ID |
| Saved preference | Use two bullets in future briefings. | By user ID |
| Current data | March sales total $2,700. | From the sales tool when preparing the answer |

The next cell seeds a practice conversation. The names `priya` and `arun` stand in for user identities. In a deployed app, get the identity from authentication and check who owns the conversation.

In [ ]:
# The application looks up a chat using the user and conversation together.
conversations = {
    ("priya", "monday"): {
        "report_month": "March",
        "comparison_month": "February",
        "messages": [
            "Can we move the meeting to 10?",
            "Yes. I'll update the calendar myself.",
            "Prepare the March sales briefing and compare it with February.",
            "Include the top March region for the leadership team.",
            "Please keep the briefing short.",
            "Write the briefing now.",
        ],
    }
}

print("Opened chat:", conversations[("priya", "monday")]["report_month"])
print("New chat:", conversations.get(("priya", "new-chat"), {}))
print("Arun's view of that chat ID:", conversations.get(("arun", "monday"), {}))

The lookup key is `(user_id, conversation_id)`. It returns March for Priya's Monday chat. It returns an empty result for her new chat and for Arun's lookup with the same conversation ID.

Save Priya's confirmed format in SQLite. Close the connection after writing, then reopen it to read the preference. This notebook uses a temporary database file.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
from tempfile import TemporaryDirectory

demo_folder = TemporaryDirectory()
profile_db = Path(demo_folder.name) / "profiles.sqlite"

with closing(sqlite3.connect(profile_db)) as db:
    db.execute("CREATE TABLE preferences (user_id TEXT PRIMARY KEY, format TEXT)")
    # Save after the app receives an explicit "remember this" choice.
    db.execute("INSERT INTO preferences VALUES (?, ?)", ("priya", "two bullets"))
    db.commit()


def load_format(user_id):
    # Open a new connection to retrieve the saved preference for this user.
    with closing(sqlite3.connect(profile_db)) as db:
        row = db.execute(
            "SELECT format FROM preferences WHERE user_id = ?", (user_id,)
        ).fetchone()
    return row[0] if row else "a short paragraph"


print("Priya in a new chat:", load_format("priya"))
print("Arun in a new chat:", load_format("arun"))

Priya's preference is available across chats. Arun gets the default. If Priya asks for a paragraph **this time**, add that instruction to the current request. If she says **from now on**, update her saved preference.

## 3. Choose the context for the briefing

**Context engineering** is choosing what to include in this model request. Load the current task, months, format, and relevant conversation messages. Then fetch the two months' totals and the leading March region through the API.

Look at the team notes in the next cell. Which one affects how you should describe March sales?

In [ ]:
sales_notes = [
    {"month": "March", "topic": "sales", "text": "The March report excludes returns."},
    {"month": "March", "topic": "calendar", "text": "The meeting starts at 10."},
    {"month": "February", "topic": "sales", "text": "February used the standard reporting process."},
    {"month": "April", "topic": "sales", "text": "The April numbers are still being collected."},
]

chat = conversations[("priya", "monday")]
month = chat["report_month"]
compare_with = chat["comparison_month"]

evidence = [
    ask_api("/report", f"Compare total sales in {compare_with} and {month}."),
    ask_api("/chat", f"Which region had the highest sales in {month}?"),
]

# Send only the sales note for the month the briefing is about.
relevant_notes = [
    note["text"] for note in sales_notes
    if note["month"] == month and note["topic"] == "sales"
]

for finding in evidence:
    print(finding)
print("Reporting note:", relevant_notes[0])

Check the results against the sales report from Lesson 02. February totals $1,800, March totals $2,700, and North led March with $1,200. The reporting note adds that the March report excludes returns.

This example passes the replies from the agents in as evidence. If you need to validate exact figures, pass structured tool results and check the amounts before displaying the final answer.

In [ ]:
import json

briefing_context = {
    "task": chat["messages"][-1],
    "report_month": month,
    "comparison_month": compare_with,
    "format": load_format("priya"),
    "recent_messages": chat["messages"][-3:-1],
    "sales_evidence": evidence,
    "reporting_notes": relevant_notes,
}

full_context = {
    **briefing_context,
    "recent_messages": chat["messages"][:-1],
    "reporting_notes": sales_notes,
}

full_text = json.dumps(full_context, indent=2)
selected_text = json.dumps(briefing_context, indent=2)

print("Full input:", len(full_text), "characters")
print("Selected input:", len(selected_text), "characters")
print(selected_text)

Compare the two payloads. The selected version leaves out the calendar note and the older messages. The earlier decision about February is still present in `comparison_month`. Check for the task, both months, the format, and the report evidence before sending the payload. The example measures characters with `len()`. A model's context limit is measured in tokens.

## 4. Generate the briefing

Pass the selected JSON to Qwen. The model sees the task, saved format, and evidence in one request. The notebook calls the two API agents first, then calls the briefing model directly so you can inspect its full input.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
    client_kwargs={"timeout": 40},
)
instructions = """Write a short sales briefing using the supplied evidence.
Include both month totals, the leading region and amount, and the reporting note. Keep the scope of the note as written.
When the format is two bullets, output exactly two separate Markdown lines.
Start each line with a dash and a space. Use this layout:
- Total sales: [comparison month and total], [report month and total].
- Leading region: [region and amount]. [Reporting note.]
When the format is a paragraph, write one short paragraph.
"""

brief = model.invoke([("system", instructions), ("human", selected_text)]).text
print(brief)

Check for two bullets. The first should give both month totals. The second should name North, give its amount, and mention the note about returns.

The **agent harness** is the code around the model that prepares its context, calls tools, handles failures, and decides when a run stops. Here, the notebook loads the chat and profile and calls the agents. `run_agent` in [sales_agent_api.py](sales_agent_api.py) limits graph steps and wait time and records which tools ran.

Behind a load balancer, a later request can land on a different container. Use shared storage when state needs to survive that move or a restart. A graph checkpointer can save state under a thread ID. The app still checks which user can access that thread. [LangChain memory overview](https://docs.langchain.com/oss/python/concepts/memory)

## 5. Check what the app remembered

Check Arun's access, Priya's saved format, the selected notes, and the final reply. The text check looks for the expected facts and counts the bullets. A stricter check would also verify which amount belongs to which month.

In [ ]:
assert conversations.get(("arun", "monday")) is None
assert conversations.get(("priya", "new-chat")) is None
assert load_format("priya") == "two bullets"
assert load_format("arun") == "a short paragraph"
print("PASS | Chat state and saved preferences have the expected scope.")

assert briefing_context["comparison_month"] == "February"
assert briefing_context["reporting_notes"] == ["The March report excludes returns."]
assert "update the calendar" not in selected_text.lower()
print("PASS | The context keeps the earlier decision and the relevant note.")

# Report model failures so we can inspect them and continue with the exercise.
answer = brief.lower().replace(",", "")
expected_facts = ("february", "1800", "march", "2700", "north", "1200", "returns")
missing_facts = [fact for fact in expected_facts if fact not in answer]
if missing_facts:
    print("FAIL | Missing facts:", missing_facts)
else:
    print("PASS | The briefing mentions the expected months, amounts, and note.")

bullets = [line for line in brief.splitlines() if line.strip().startswith("- ")]
if len(bullets) == 2:
    print("PASS | The briefing uses Priya's saved two-bullet format.")
else:
    print("FAIL | Expected two bullets. Found", len(bullets))


**Try it.** Priya asks for a paragraph for this briefing only. Create a new context with `{**briefing_context, "format": "one paragraph"}`, then predict what `load_format("priya")` will return. The next cell checks that request and simulates her later asking to forget the saved preference.

In [ ]:
# Apply a one-time instruction to this request only.
one_time_context = {**briefing_context, "format": "one paragraph"}
assert one_time_context["format"] == "one paragraph"
assert load_format("priya") == "two bullets"

# Later, Priya explicitly asks the app to forget her saved format.
with closing(sqlite3.connect(profile_db)) as db:
    db.execute("DELETE FROM preferences WHERE user_id = ?", ("priya",))
    db.commit()

assert load_format("priya") == "a short paragraph"
print("PASS | A one-time change leaves the profile alone. Forgetting removes the saved format.")

**Try a fresh chat.** Priya asks, “Do the same for April.” Decide what to ask if the comparison month is missing. Then call `/report` with April and March and check how it handles the missing April data.

Next, open the [invoice case study](06_invoice_review_agent_case_study.ipynb). [Effective context engineering](https://www.anthropic.com/research/effective-context-engineering-for-ai-agents)